In [1]:
!pip install transformers datasets accelerate torch -q

In [2]:
!pip install --upgrade transformers

In [3]:
import transformers
print(transformers.__version__)

5.2.0


In [4]:
import torch
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

In [5]:
dataset = load_dataset("euclaise/writingprompts")
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'story'],
        num_rows: 272600
    })
    test: Dataset({
        features: ['prompt', 'story'],
        num_rows: 15138
    })
    validation: Dataset({
        features: ['prompt', 'story'],
        num_rows: 15620
    })
})

In [6]:
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
small_val_dataset = dataset["validation"].shuffle(seed=42).select(range(2000))

In [7]:
def combine_text(example):
    example["text"] = example["prompt"] + "\n" + example["story"]
    return example

small_train_dataset = small_train_dataset.map(combine_text)
small_val_dataset = small_val_dataset.map(combine_text)

In [8]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [9]:
prompt = "A mysterious door appears in the middle of the city."

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs["input_ids"],
    max_length=300,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

print("===== BEFORE FINE-TUNING =====\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


===== BEFORE FINE-TUNING =====

A mysterious door appears in the middle of the city. One of the guards asks the man if he has any weapons, and he says he has two. He says that he is a fighter, and that he will fight back, but one of the guards says he's not going to let him escape with his weapon. The others ask him if he wants to help them. He says that he will not do it, and the guard tells him that he can't say that.

There are several other guards with similar names, but he's not looking for them. He says that he's going to save them all, but he can't have the other guards get hurt. He asks them where the others are going, but the guards say that they are going to fight the thieves. The guards ask him what they will do with them, but the man says that they are going to let them live, and they are going to fight back.

Meanwhile, the guards have just joined the rebellion and are fighting with the others. The guards say that the thieves are going to kill them all, and that they are g

In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_train = small_train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=small_train_dataset.column_names
)

tokenized_val = small_val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=small_val_dataset.column_names
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-writingprompts",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    logging_steps=200,
    save_steps=1000,
    save_total_limit=2,
    report_to=[]   # ✅ FIXED
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

c:\Users\hp\.anaconda\conda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./fine_tuned_gpt2_writingprompts")
tokenizer.save_pretrained("./fine_tuned_gpt2_writingprompts")

In [ ]:
prompt = "A mysterious door appears in the middle of the city."

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs["input_ids"],
    max_length=300,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))